In [17]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import json
import time

In [2]:
dataset_a = [
    {"username": "alice_smith", "email": "alice@example.com", "age": 28, "signup_date": "2023-10-01"},
    {"username": "bob_jones", "email": "bob.jones@example.com", "age": 34, "signup_date": "2023-10-02"},
    {"username": "charlie_davis", "email": "charlie@example.com", "age": 22, "signup_date": "2023-10-03"},
    {"username": "diana_prince", "email": "diana@example.com", "age": 30, "signup_date": "2023-10-04"},
    {"username": "evan_wright", "email": "evan@example.com", "age": 45, "signup_date": "2023-10-05"},
    {"username": "frank_castle", "email": "frankcastle.com", "age": 38, "signup_date": "2023-10-06"},
    {"username": "grace_hopper", "email": "grace@example.com", "age": "22", "signup_date": "2023-10-07"},
    {"username": "", "email": "unknown.user@example.com", "age": 19, "signup_date": "2023-10-08"}
]

data rep: mix of user input and sys generated because signup date is not something which user inputs. All the other datas are user input.

Quality issue: issue is likely might be wrong datas about users, which is users might be lying about their personal infos like age or something

validations: checking type of ages

In [3]:
dataset_b = [
    {"timestamp": "2023-10-10T08:00:01Z", "service": "AuthService", "level": "INFO", "message": "User login successful."},
    {"timestamp": "2023-10-10T08:05:23Z", "service": "PaymentAPI", "level": "INFO", "message": "Payment processing initiated."},
    {"timestamp": "2023-10-10T08:05:45Z", "service": "PaymentAPI", "level": "WARNING", "message": "High latency detected from payment gateway."},
    {"timestamp": "2023-10-10T08:06:12Z", "service": "DatabaseService", "level": "INFO", "message": "Database backup completed successfully."},
    {"timestamp": "2023-10-10T08:15:00Z", "service": "AuthService", "level": "WARNING", "message": "Multiple failed login attempts for user ID 402."},
    {"timestamp": "2023-10-10T08:17:33Z", "service": "PaymentAPI", "level": "ERROR", "message": "Payment gateway timeout. Transaction failed."},
    {"timestamp": "2023-10-10T08:20:05Z", "service": "DatabaseService", "level": "ERROR", "message": "Connection pool exhausted."},
    {"timestamp": "2023-10-10T08:22:10Z", "service": "AuthService", "level": "INFO", "message": "Password reset email sent to user ID 402."},
    {"timestamp": "2023-10-10T08:30:00Z", "service": "DatabaseService", "level": "INFO", "message": "Connection pool restored."},
    {"timestamp": "2023-10-10T08:45:11Z", "service": "PaymentAPI", "level": "INFO", "message": "Payment processing retry successful."}
]

data rep: system generated

quality issue:error and warning messages are too short to interpret and understand the problem, timestamps does not include milli or microseconds, which is crucial for large databases which gets hundreds of loggings each second. and there is no way to trace users from message and services, if payment is failed or illegally large for one users, we cant find who is this user

validation: add micro and milli seconds to timestamps

In [4]:
dataset_c = [
    {"sku": "LAP-X1-16GB", "product_name": "ProBook X1 Laptop", "quantity_in_stock": 145, "warehouse_location": "Zone A - NY", "last_updated": "2023-10-09"},
    {"sku": "MON-27-4K", "product_name": "UltraView 27-inch 4K Monitor", "quantity_in_stock": 32, "warehouse_location": "Zone B - CA", "last_updated": "2023-10-08"},
    {"sku": "KBD-MECH-RGB", "product_name": "ClickMaster Mechanical Keyboard", "quantity_in_stock": 0, "warehouse_location": "Zone A - NY", "last_updated": "2023-10-05"},
    {"sku": "MSE-WIR-PRO", "product_name": "GlidePro Wireless Mouse", "quantity_in_stock": 210, "warehouse_location": "Zone C - TX", "last_updated": "2023-10-09"},
    {"sku": "CBL-USB-C-2M", "product_name": "Braided USB-C Cable (2m)", "quantity_in_stock": 850, "warehouse_location": "Zone B - CA", "last_updated": "2023-10-01"},
    {"sku": "DOCK-STN-UNV", "product_name": "Universal USB-C Docking Station", "quantity_in_stock": 15, "warehouse_location": "Zone C - TX", "last_updated": "2023-10-10"}
]

data rep: internal database of a company

quality issue: last update is not enough precise, if a customer would buy product in a same day, stock will not be updated till tomorrow.

validation: add hours to the last update, split location to two parts, zone and state, check if stocks are numeric type

In [ ]:
def validate_registrations(entries):
    valid_entries = []
    invalid_entries = []

    for entry in entries:
        is_valid = True
        
        username = entry.get("username", "")
        if not username or str(username).strip() == "":
            is_valid = False
            
        email = entry.get("email", "")
        if "@" not in str(email):
            is_valid = False
            
        try:
            age = int(entry.get("age"))
            if age <= 0:
                is_valid = False
        except (ValueError, TypeError):
            is_valid = False
        
        if is_valid:
            valid_entries.append(entry)
        else:
            invalid_entries.append(entry)
            
    return valid_entries, invalid_entries

valid, invalid = validate_registrations(dataset_a)

print(f"Total entries processed: {len(dataset_a)}")
print(f"Valid entries count: {len(valid)}")
print(f"Invalid entries count: {len(invalid)}")

Total entries processed: 8
Valid entries count: 6
Invalid entries count: 2


In [6]:
np.random.seed(42)

rows = 500
rides=pd.DataFrame({
    'ride_id': range(1,rows+1),
    'driver_id': np.random.randint(1000,1051,size=rows),
    'distance_km': np.random.uniform(1.0,30.0,size=rows).round(2),
    'fare_usd': np.random.uniform(5.0,75.0, size=rows).round(2),
    'ride_type': np.random.choice(['standard','premium','pool'],size=rows),
    'city': np.random.choice(['Berlin','Seoul','Nairobi','Toronto','Lima'],size=rows),
    'timestamps': pd.date_range(start='2025-01-01',periods=rows,freq='min')
})
rides = rides.set_index('ride_id')

rides.head()


,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00


In [7]:
rides.to_csv('rides_csv',index=True)

In [8]:
rides.reset_index().to_json('rides.jsonl',orient="records", lines=True,date_format='iso')

In [9]:
rides.to_parquet('rides.parquet')

In [10]:
import os

json_size = os.path.getsize("rides.jsonl")/1024
pq_size = os.path.getsize("rides.parquet")/1024
csv_size = os.path.getsize("rides_csv")/1024

print("json size(in KBs): ",json_size)
print("pq size(in KBs): ",pq_size)
print("csv size(in KBs): ",csv_size)

json size(in KBs):  71.0
pq size(in KBs):  17.5703125
csv size(in KBs):  27.119140625


In [11]:
csv_df = pd.read_csv('rides_csv',index_col=0)
csv_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [12]:
csv_df.shape

(500, 6)

In [13]:
json_df = pd.read_json('rides.jsonl',lines=True).set_index('ride_id')
json_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [14]:
json_df.shape

(500, 6)

In [15]:
parquet_df = pd.read_parquet('rides.parquet')
parquet_df

,driver_id,distance_km,fare_usd,ride_type,city,timestamps
ride_id,,,,,,
1,1038,3.66,45.43,standard,Berlin,2025-01-01 00:00:00
2,1028,3.73,16.59,pool,Seoul,2025-01-01 00:01:00
3,1014,10.03,7.37,pool,Seoul,2025-01-01 00:02:00
4,1042,29.41,26.80,premium,Lima,2025-01-01 00:03:00
5,1007,6.08,59.63,pool,Nairobi,2025-01-01 00:04:00
...,...,...,...,...,...,...
496,1050,27.64,8.06,pool,Nairobi,2025-01-01 08:15:00
497,1036,18.02,16.32,premium,Nairobi,2025-01-01 08:16:00
498,1021,1.95,6.46,standard,Berlin,2025-01-01 08:17:00


In [16]:
parquet_df.shape

(500, 6)

As a matter of readability, in my opinion, csv format is best but as a matter of memory, parquet is best, each of them preserved data nicely.

In [23]:
df = pd.DataFrame(np.random.randn(200000,20),columns=[f"feature_{i}" for i in range(20)])
np_arr = df.to_numpy()
df

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19
0,-0.028338,0.521842,-1.434644,-1.261863,-2.462930,-1.059584,-0.021050,-0.585669,-1.539731,-0.196932,-0.478900,-0.056695,0.364907,1.162371,0.138973,-0.426452,0.163896,-0.154964,1.796673,-0.821014
1,-1.007751,-0.669252,-0.461278,1.461981,0.274902,-0.938360,-1.818677,-0.685499,0.644994,-0.335113,-0.837785,-0.368472,-0.464672,-1.604226,-0.172537,-0.205188,1.624013,0.024197,0.662705,-0.414552
2,0.029967,1.288090,-0.451289,0.645329,-0.155564,-0.154303,-1.079704,1.786594,1.046450,0.750704,-0.240975,1.067266,0.969839,-1.041183,-0.554105,-0.123087,1.143429,1.147621,-0.683828,0.118065
3,0.426091,-0.974864,1.194441,1.057515,-0.441438,0.788220,0.207483,0.025585,0.603758,-0.809574,0.564645,0.742203,-0.381677,-1.302949,1.074671,-0.748906,-1.861617,1.521357,-0.071414,-1.064988
4,0.136975,1.013419,-0.528143,-1.216618,-1.195990,0.651788,0.486885,0.136565,-0.470521,0.726499,-2.926333,-2.481397,-1.315013,2.102571,-1.651371,0.661639,-0.942419,-1.059687,-0.589065,0.660924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,-1.142694,1.192154,-0.562659,0.505512,2.036002,-0.917407,0.566835,0.506691,1.393844,-0.386571,-0.402976,-0.211791,-2.037697,1.178225,0.702137,-0.106649,0.739832,0.300888,0.889035,0.111580
199996,0.024092,-0.920660,-1.280819,0.224128,-0.886825,-0.291770,-0.175037,-0.894578,-0.174176,0.462717,-2.265709,0.625406,0.266651,-0.504056,-0.762038,-0.431073,-0.644996,-0.352428,0.869992,0.937001
199997,0.707041,0.408002,0.234171,-0.023476,-1.046394,-1.428356,1.767631,-0.081271,1.467754,-0.019971,0.106464,0.598554,0.515063,0.543432,-0.463993,0.189762,0.550703,-0.641530,-1.308009,-0.244875
199998,-0.197701,1.141372,0.766053,0.117320,2.156849,0.422803,1.170290,-0.675714,0.165242,1.288872,-0.106661,0.140649,-1.371928,1.916494,-0.136655,-0.908665,-2.999803,-1.505706,1.014972,0.742787


In [40]:
%timeit df["feature_0"].mean()

1.66 ms ± 118 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
def row_sum(df):
    total = 0
    for i in range (10000):
        total+=df.iloc[i]["feature_0"]
    return total
%timeit row_sum(df)


153 ms ± 6.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [44]:
%timeit np_arr[:,0].mean()

697 μs ± 44.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [45]:
def np_row_sum(np_arr):
    total=0
    for i in range(20000):
        total+=np_arr[i,0]
    return total
%timeit np_row_sum(np_arr)

2.98 ms ± 160 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


pandas is column-major, that is why iterating by rows using iloc takes so much time, each time pandas should look up for an index and sum them because each row elements point different memory location. at the same time, accessing just one column (feature_0) is significantly faster because pandas pushes all the elements of column at once. 

numpy is even faster than pandas because numpy does not use complex indexing and labeling, so, though numpy is row-major, executing columnal operations is significantly faster than pandas

numpy lay out data more simpler, as a raw data without indexing and labeling. pandas is more complex with indices and labels

In [ ]:
np.random.seed(42)
num_rows = 100000
df = pd.DataFrame(np.random.randn(num_rows, 5), columns=[f'col_{i}' for i in range(5)])
df['category'] = np.random.choice(["alpha", "beta", "gamma"], size=num_rows)

df.to_csv("data.csv", index=False)
df.to_parquet("data.parquet", index=False)

csv_size = os.path.getsize("data.csv") / 1024
parquet_size = os.path.getsize("data.parquet") / 1024

print(f"CSV Size: {csv_size:.2f} KB")
print(f"Parquet Size: {parquet_size:.2f} KB")

CSV Size: 10236.52 KB
Parquet Size: 4973.53 KB


In [ ]:
csv_df = pd.read_csv("data.csv")
parquet_df = pd.read_parquet("data.parquet")

print("CSV Dtypes:\n", csv_df.dtypes)
print("\nParquet Dtypes:\n", parquet_df.dtypes)

CSV Dtypes:
 col_0       float64
col_1       float64
col_2       float64
col_3       float64
col_4       float64
category     object
dtype: object

Parquet Dtypes:
 col_0       float64
col_1       float64
col_2       float64
col_3       float64
col_4       float64
category     object
dtype: object


In [48]:
start = time.time()
pd.read_csv("data.csv")
csv_time = time.time() - start

start = time.time()
pd.read_parquet("data.parquet")
parquet_time = time.time() - start

print(f"CSV Read Time: {csv_time:.4f} seconds")
print(f"Parquet Read Time: {parquet_time:.4f} seconds")

CSV Read Time: 0.0939 seconds
Parquet Read Time: 0.0122 seconds


In [49]:
start = time.time()
category_only = pd.read_parquet("data.parquet", columns=["category"])
parquet_selective_time = time.time() - start

print(f"Parquet Selective Read Time: {parquet_selective_time:.4f} seconds")

Parquet Selective Read Time: 0.0085 seconds


CSV is a row-major, so if we want to extract any specific column, machine has to check every index and element located in this column, but parquet is column-major so we can easily read and extract this exact column

In [50]:
flat_data = [
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "format": "Paperback", "publisher_name": "Scribner", "publisher_country": "USA", "price": 15.99},
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "format": "E-book", "publisher_name": "Scribner", "publisher_country": "USA", "price": 9.99},
    {"title": "1984", "author": "George Orwell", "format": "Paperback", "publisher_name": "Secker & Warburg", "publisher_country": "UK", "price": 12.50},
    {"title": "1984", "author": "George Orwell", "format": "E-book", "publisher_name": "Secker & Warburg", "publisher_country": "UK", "price": 7.99},
    {"title": "The Hobbit", "author": "J.R.R. Tolkien", "format": "Paperback", "publisher_name": "Allen & Unwin", "publisher_country": "UK", "price": 18.00},
    {"title": "The Hobbit", "author": "J.R.R. Tolkien", "format": "E-book", "publisher_name": "Allen & Unwin", "publisher_country": "UK", "price": 11.00},
    {"title": "Foundation", "author": "Isaac Asimov", "format": "Paperback", "publisher_name": "Gnome Press", "publisher_country": "USA", "price": 14.50},
    {"title": "Dune", "author": "Frank Herbert", "format": "Paperback", "publisher_name": "Chilton Books", "publisher_country": "USA", "price": 22.00},
    {"title": "Dune", "author": "Frank Herbert", "format": "E-book", "publisher_name": "Chilton Books", "publisher_country": "USA", "price": 12.99},
    {"title": "Beloved", "author": "Toni Morrison", "format": "Paperback", "publisher_name": "Alfred A. Knopf", "publisher_country": "USA", "price": 16.00},
    {"title": "Never Let Me Go", "author": "Kazuo Ishiguro", "format": "Paperback", "publisher_name": "Faber & Faber", "publisher_country": "UK", "price": 13.99},
    {"title": "Norwegian Wood", "author": "Haruki Murakami", "format": "Paperback", "publisher_name": "Kodansha", "publisher_country": "Japan", "price": 17.50}
]

df_flat = pd.DataFrame(flat_data)

In [51]:
publishers_data = [
    {"publisher_id": 1, "publisher_name": "Scribner", "publisher_country": "USA"},
    {"publisher_id": 2, "publisher_name": "Secker & Warburg", "publisher_country": "UK"},
    {"publisher_id": 3, "publisher_name": "Allen & Unwin", "publisher_country": "UK"},
    {"publisher_id": 4, "publisher_name": "Gnome Press", "publisher_country": "USA"},
    {"publisher_id": 5, "publisher_name": "Chilton Books", "publisher_country": "USA"},
    {"publisher_id": 6, "publisher_name": "Alfred A. Knopf", "publisher_country": "USA"},
    {"publisher_id": 7, "publisher_name": "Faber & Faber", "publisher_country": "UK"},
    {"publisher_id": 8, "publisher_name": "Kodansha", "publisher_country": "Japan"}
]
df_publishers = pd.DataFrame(publishers_data)

books_data = [
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "format": "Paperback", "publisher_id": 1, "price": 15.99},
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "format": "E-book", "publisher_id": 1, "price": 9.99},
    {"title": "1984", "author": "George Orwell", "format": "Paperback", "publisher_id": 2, "price": 12.50},
    {"title": "1984", "author": "George Orwell", "format": "E-book", "publisher_id": 2, "price": 7.99},
    {"title": "The Hobbit", "author": "J.R.R. Tolkien", "format": "Paperback", "publisher_id": 3, "price": 18.00},
    {"title": "The Hobbit", "author": "J.R.R. Tolkien", "format": "E-book", "publisher_id": 3, "price": 11.00},
    {"title": "Foundation", "author": "Isaac Asimov", "format": "Paperback", "publisher_id": 4, "price": 14.50},
    {"title": "Dune", "author": "Frank Herbert", "format": "Paperback", "publisher_id": 5, "price": 22.00},
    {"title": "Dune", "author": "Frank Herbert", "format": "E-book", "publisher_id": 5, "price": 12.99},
    {"title": "Beloved", "author": "Toni Morrison", "format": "Paperback", "publisher_id": 6, "price": 16.00},
    {"title": "Never Let Me Go", "author": "Kazuo Ishiguro", "format": "Paperback", "publisher_id": 7, "price": 13.99},
    {"title": "Norwegian Wood", "author": "Haruki Murakami", "format": "Paperback", "publisher_id": 8, "price": 17.50}
]
df_books = pd.DataFrame(books_data)

df_verified = pd.merge(df_books, df_publishers, on="publisher_id").drop(columns=['publisher_id'])

In [53]:
doc_store = [
    {
        "title": "The Great Gatsby",
        "author": "F. Scott Fitzgerald",
        "publisher": {"name": "Scribner", "country": "USA"},
        "editions": [
            {"format": "Paperback", "price": 15.99},
            {"format": "E-book", "price": 9.99}
        ]
    },
    {
        "title": "1984",
        "author": "George Orwell",
        "publisher": {"name": "Secker & Warburg", "country": "UK"},
        "editions": [
            {"format": "Paperback", "price": 12.50},
            {"format": "E-book", "price": 7.99}
        ]
    },
    {
        "title": "The Hobbit",
        "author": "J.R.R. Tolkien",
        "publisher": {"name": "Allen & Unwin", "country": "UK"},
        "editions": [
            {"format": "Paperback", "price": 18.00},
            {"format": "E-book", "price": 11.00}
        ]
    },
    {
        "title": "Foundation",
        "author": "Isaac Asimov",
        "publisher": {"name": "Gnome Press", "country": "USA"},
        "editions": [
            {"format": "Paperback", "price": 14.50}
        ]
    },
    {
        "title": "Dune",
        "author": "Frank Herbert",
        "publisher": {"name": "Chilton Books", "country": "USA"},
        "editions": [
            {"format": "Paperback", "price": 22.00},
            {"format": "E-book", "price": 12.99}
        ]
    },
    {
        "title": "Beloved",
        "author": "Toni Morrison",
        "publisher": {"name": "Alfred A. Knopf", "country": "USA"},
        "editions": [
            {"format": "Paperback", "price": 16.00}
        ]
    },
    {
        "title": "Never Let Me Go",
        "author": "Kazuo Ishiguro",
        "publisher": {"name": "Faber & Faber", "country": "UK"},
        "editions": [
            {"format": "Paperback", "price": 13.99}
        ]
    },
    {
        "title": "Norwegian Wood",
        "author": "Haruki Murakami",
        "publisher": {"name": "Kodansha", "country": "Japan"},
        "editions": [
            {"format": "Paperback", "price": 17.50}
        ]
    }
]

In [54]:
print("--- QUESTION 1: Books by F. Scott Fitzgerald ---")
# Flat
res_flat = df_flat[df_flat['author'] == 'F. Scott Fitzgerald']['title'].unique()
print(f"Flat: {res_flat}")
# Normalized
res_norm = df_books[df_books['author'] == 'F. Scott Fitzgerald']['title'].unique()
print(f"Normalized: {res_norm}")
# Document
res_doc = [b['title'] for b in doc_store if b['author'] == 'F. Scott Fitzgerald']
print(f"Document: {res_doc}\n")

print("--- QUESTION 2: Average Price across all editions ---")
# Flat
avg_flat = df_flat['price'].mean()
print(f"Flat Average: ${avg_flat:.2f}")
# Normalized
avg_norm = df_books['price'].mean()
print(f"Normalized Average: ${avg_norm:.2f}")
# Document
all_prices = [ed['price'] for b in doc_store for ed in b['editions']]
avg_doc = sum(all_prices) / len(all_prices)
print(f"Document Average: ${avg_doc:.2f}\n")

print("--- QUESTION 3: Update Scribner country to Canada ---")
# Flat Update
before_flat = len(df_flat[df_flat['publisher_name'] == 'Scribner'])
df_flat.loc[df_flat['publisher_name'] == 'Scribner', 'publisher_country'] = 'Canada'
print(f"Flat: Changed {before_flat} records.")

# Normalized Update
before_norm = len(df_publishers[df_publishers['publisher_name'] == 'Scribner'])
df_publishers.loc[df_publishers['publisher_name'] == 'Scribner', 'publisher_country'] = 'Canada'
print(f"Normalized: Changed {before_norm} record (The single source of truth).")

# Document Update
count_doc = 0
for b in doc_store:
    if b['publisher']['name'] == 'Scribner':
        b['publisher']['country'] = 'Canada'
        count_doc += 1
print(f"Document: Changed {count_doc} documents.")

--- QUESTION 1: Books by F. Scott Fitzgerald ---
Flat: ['The Great Gatsby']
Normalized: ['The Great Gatsby']
Document: ['The Great Gatsby']

--- QUESTION 2: Average Price across all editions ---
Flat Average: $14.37
Normalized Average: $14.37
Document Average: $14.37

--- QUESTION 3: Update Scribner country to Canada ---
Flat: Changed 2 records.
Normalized: Changed 1 record (The single source of truth).
Document: Changed 1 documents.


The flat representation made answering the first two questions easiest because all data is available in a single row. The normalized representation is best for the third question because updating the publisher's country only requires changing one record, ensuring consistency.

I would choose Normalized if publisher info changes frequently to maintain a single source of truth. I would choose Document if each book is accessed as a standalone record to avoid joins.